# Fossil Looker — Mini-Projeto 2 (Fuzzy Edition)

---

**Fossil Looker** é um sistema classificador de dinossauros. A intenção é que, baseado em características visíveis em fósseis (e imagens de dinossauros, no geral), qualquer usuário consiga identificar dinossauros através de características linguísticas fuzzy.

Essa é a versão **Mini-Projeto 2**, onde substituímos as regras IF–THEN nítidas (*crisp*) do MP1 por **inferência fuzzy Mamdani** usando *scikit-fuzzy*. Em vez de respostas binárias (`afiados` ou `achatados`), o usuário agora fornece graus pra cada característica: o sistema tolera incerteza, o que faz muito mais sentido quando se está olhando para um fóssil de 66 milhões de anos.

O elenco também cresceu: saímos de **8 dinossauros** para **15**. Bem-vindo ao Jurássico aumentado.

---

### Dependências

```bash
pip install scikit-fuzzy numpy matplotlib
```

In [ ]:
# Dependências do projeto — scikit-fuzzy para a inferência Mamdani
# numpy e matplotlib são arrastados junto de qualquer jeito, mas vamos ser explícitos
!pip install scikit-fuzzy numpy matplotlib -q

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')  # skfuzzy às vezes reclama de coisas que funcionam perfeitamente

## Arquitetura do Sistema Fuzzy

O **Fossil Looker Fuzzy** se baseia em um controlador **Mamdani** com **4 entradas** e **1 saída**.

### Variáveis Linguísticas de Entrada

| Variável | Escala | Termos Linguísticos | Justificativa |
|---|---|---|---|
| `dentes` | 0–10 | *afiados*, *mistos*, *achatados* | Morfologia dental é o principal indicador de dieta; escala contínua reflete desgaste e variação fóssil |
| `locomocao` | 0–10 | *bipede*, *misto*, *quadrupede* | Alguns dinos como Iguanodon alternavam entre os dois modos — o fuzzy captura isso |
| `appendice` | 0–10 | *chifres*, *crista*, *dorsal* | Apêndices cranianos e dorsais são os principais traços de identificação de gênero |
| `tamanho` | 0–10 | *pequeno*, *medio*, *grande* | Tamanho do espécime ajuda a distinguir entre gêneros próximos |

### Variável de Saída

| Variável | Escala | Termos |
|---|---|---|
| `especie` | 0–14 | Um termo por dinossauro (15 no total) |

Cada dinossauro ocupa uma "fatia" do universo de saída. A defuzzificação pelo método **centroide** devolve o índice do dinossauro mais provável dado o conjunto de entradas.

## Definição das Variáveis e Funções de Pertinência

In [ ]:
# -------------------------------------------------------------------------
# UNIVERSOS DE DISCURSO
# Todas as entradas usam escala 0–10 para facilitar a interface com o usuário
# -------------------------------------------------------------------------

dentes    = ctrl.Antecedent(np.linspace(0, 10, 200), 'dentes')
locomocao = ctrl.Antecedent(np.linspace(0, 10, 200), 'locomocao')
appendice = ctrl.Antecedent(np.linspace(0, 10, 200), 'appendice')
tamanho   = ctrl.Antecedent(np.linspace(0, 10, 200), 'tamanho')

# A saída vai de 0 a 14: cada inteiro corresponde a um dinossauro
especie   = ctrl.Consequent(np.linspace(0, 14, 280), 'especie')

print("Variáveis criadas. Universos definidos.")

In [ ]:
# -------------------------------------------------------------------------
# FUNÇÕES DE PERTINÊNCIA — DENTES
#
# 0 = completamente afiados (carnívoro puro, ex: T-Rex)
# 5 = mistos (ex: algumas espécies onívoras ou juvenis)
# 10 = completamente achatados (herbívoro puro, ex: Triceratops)
#
# Justificativa: dentes são o traço diagnóstico primário de dieta.
# A sobreposição em 'mistos' reflete que fósseis degradados raramente
# revelam dentição perfeita — incerteza é esperada.
# -------------------------------------------------------------------------

dentes['afiados']  = fuzz.trimf(dentes.universe, [0, 0, 5])
dentes['mistos']   = fuzz.trimf(dentes.universe, [2, 5, 8])
dentes['achatados'] = fuzz.trimf(dentes.universe, [5, 10, 10])

# -------------------------------------------------------------------------
# FUNÇÕES DE PERTINÊNCIA — LOCOMOÇÃO
#
# 0 = completamente bípede
# 5 = facultativo (alternava entre dois e quatro membros)
# 10 = completamente quadrúpede
#
# Justificativa: Iguanodon e Hadrossauros eram facultativamente bípedes.
# A escala fuzzy captura isso de forma elegante — o MP1 ignorava essa nuance.
# -------------------------------------------------------------------------

locomocao['bipede']     = fuzz.trimf(locomocao.universe, [0, 0, 5])
locomocao['misto']      = fuzz.trimf(locomocao.universe, [2, 5, 8])
locomocao['quadrupede'] = fuzz.trimf(locomocao.universe, [5, 10, 10])

# -------------------------------------------------------------------------
# FUNÇÕES DE PERTINÊNCIA — APÊNDICE
#
# 0–3 = chifres (estruturas ósseas projetadas para a frente)
# 3–7 = crista (estruturas na cabeça, tubulares ou laminares)
# 7–10 = dorsal (placas, velas ou espinhos ao longo do dorso)
#
# Justificativa: apêndices são traços altamente distintivos entre gêneros.
# A separação em três regiões é suficiente para cobrir os 15 dinossauros
# sem sobreposição excessiva.
# -------------------------------------------------------------------------

appendice['chifres'] = fuzz.trimf(appendice.universe, [0, 0, 4])
appendice['crista']  = fuzz.trimf(appendice.universe, [2, 5, 8])
appendice['dorsal']  = fuzz.trimf(appendice.universe, [6, 10, 10])

# -------------------------------------------------------------------------
# FUNÇÕES DE PERTINÊNCIA — TAMANHO
#
# 0–4 = pequeno (< 3m de comprimento: Velociraptor, Carnotaurus juvenil)
# 3–7 = médio (3–8m: Allosaurus, Parasaurolophus)
# 6–10 = grande (> 8m: T-Rex, Spinosaurus, Brachiosaurus)
#
# Justificativa: espécies como Velociraptor e T-Rex são facilmente
# confundíveis em características gerais — o tamanho é o desambiguador.
# -------------------------------------------------------------------------

tamanho['pequeno'] = fuzz.trimf(tamanho.universe, [0, 0, 4])
tamanho['medio']   = fuzz.trimf(tamanho.universe, [2, 5, 8])
tamanho['grande']  = fuzz.trimf(tamanho.universe, [6, 10, 10])

print("Funções de pertinência das entradas definidas.")

In [ ]:
# -------------------------------------------------------------------------
# FUNÇÕES DE PERTINÊNCIA — SAÍDA (espécie)
#
# Cada dinossauro ocupa uma posição no universo 0–14.
# Usamos gaussmf para a saída porque suaviza a defuzzificação pelo
# centroide — picos triangulares muito estreitos podem gerar resultados
# instáveis quando mais de uma regra dispara com força similar.
#
# Roster completo (15 dinossauros):
#   TERÓPODES (carnívoros bípedes): Velociraptor, T-Rex, Spinosaurus,
#       Ceratosaurus, Cryolophosaurus, Allosaurus, Carnotaurus
#   ORNITÓPODES (herbívoros bípedes): Parasaurolophus, Iguanodon,
#       Ouranosaurus, Corythosaurus
#   OUTROS HERBÍVOROS (quadrúpedes): Triceratops, Stegosaurus,
#       Ankylosaurus, Brachiosaurus
# -------------------------------------------------------------------------

DINOS = [
    # idx  nome               grupo
    (0,  'velociraptor'),   # terópode
    (1,  'trex'),           # terópode
    (2,  'spinosaurus'),    # terópode
    (3,  'ceratosaurus'),   # terópode
    (4,  'cryolophosaurus'),# terópode
    (5,  'allosaurus'),     # terópode
    (6,  'carnotaurus'),    # terópode
    (7,  'parasaurolophus'),# ornitópode
    (8,  'iguanodon'),      # ornitópode
    (9,  'ouranosaurus'),   # ornitópode
    (10, 'corythosaurus'),  # ornitópode
    (11, 'triceratops'),    # outro herbívoro
    (12, 'stegosaurus'),    # outro herbívoro
    (13, 'ankylosaurus'),   # outro herbívoro
    (14, 'brachiosaurus'),  # outro herbívoro
]

sigma = 0.55  # largura da gaussiana — estreito o suficiente para separar dinos vizinhos

for idx, nome in DINOS:
    especie[nome] = fuzz.gaussmf(especie.universe, idx, sigma)

print(f"Saída definida: {len(DINOS)} dinossauros carregados no universo.")

## Visualização das Funções de Pertinência

Antes de escrever as regras, vale ver como as variáveis se parecem visualmente. Isso ajuda a validar se as funções fazem sentido para o domínio.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Fossil Looker — Funções de Pertinência das Entradas', fontsize=14, fontweight='bold')

# Dentes
ax = axes[0, 0]
for termo in ['afiados', 'mistos', 'achatados']:
    ax.plot(dentes.universe, dentes[termo].mf, label=termo, linewidth=2)
ax.set_title('Dentes')
ax.set_xlabel('0 = afiados  →  10 = achatados')
ax.set_ylabel('Pertinência')
ax.legend()
ax.grid(True, alpha=0.3)

# Locomoção
ax = axes[0, 1]
for termo in ['bipede', 'misto', 'quadrupede']:
    ax.plot(locomocao.universe, locomocao[termo].mf, label=termo, linewidth=2)
ax.set_title('Locomoção')
ax.set_xlabel('0 = bípede  →  10 = quadrúpede')
ax.set_ylabel('Pertinência')
ax.legend()
ax.grid(True, alpha=0.3)

# Apêndice
ax = axes[1, 0]
for termo in ['chifres', 'crista', 'dorsal']:
    ax.plot(appendice.universe, appendice[termo].mf, label=termo, linewidth=2)
ax.set_title('Apêndice')
ax.set_xlabel('0 = chifres  →  5 = crista  →  10 = dorsal')
ax.set_ylabel('Pertinência')
ax.legend()
ax.grid(True, alpha=0.3)

# Tamanho
ax = axes[1, 1]
for termo in ['pequeno', 'medio', 'grande']:
    ax.plot(tamanho.universe, tamanho[termo].mf, label=termo, linewidth=2)
ax.set_title('Tamanho')
ax.set_xlabel('0 = pequeno  →  10 = grande')
ax.set_ylabel('Pertinência')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Base de Regras Fuzzy

A base de regras segue a mesma lógica taxonômica do MP1 — três níveis de aprofundamento — mas agora expressas em termos linguísticos fuzzy.

**Total: 15 regras** — uma por dinossauro, cobrindo todas as combinações relevantes do espaço de entradas.

In [ ]:
# =========================================================================
# BASE DE REGRAS FUZZY — FOSSIL LOOKER (MAMDANI)
#
# Estrutura das regras:
#   dentes + locomocao -> grupo (terópode / ornitópode / outro herbívoro)
#   grupo + appendice + tamanho -> espécie
#
# Como o scikit-fuzzy non-Mamdani trabalha com regras planas (não hierárquicas),
# codificamos os três níveis em uma única camada de regras que combina
# diretamente as quatro entradas com a saída de espécie.
# =========================================================================

# -------------------------------------------------------------------------
# TERÓPODES — Carnívoros Bípedes
# Características de grupo: dentes=afiados, locomocao=bipede
# Diferenciadores: appendice + tamanho
# -------------------------------------------------------------------------

# R1: Afiados + Bípede + Chifres + Pequeno -> Velociraptor
# Velociraptor era pequeno (1,8m), sem apêndice craniano proeminente mas
# com garras curvadas — modelamos o 'chifre' como ausência de crista/dorsal
r1 = ctrl.Rule(
    dentes['afiados'] & locomocao['bipede'] & appendice['chifres'] & tamanho['pequeno'],
    especie['velociraptor']
)

# R2: Afiados + Bípede + Chifres + Grande -> T-Rex
# T-Rex: sem apêndice craniano, porém enorme (12m+)
r2 = ctrl.Rule(
    dentes['afiados'] & locomocao['bipede'] & appendice['chifres'] & tamanho['grande'],
    especie['trex']
)

# R3: Afiados + Bípede + Dorsal + Grande -> Spinosaurus
# Spinosaurus: vela dorsal icônica, maior terópode conhecido (14–18m)
r3 = ctrl.Rule(
    dentes['afiados'] & locomocao['bipede'] & appendice['dorsal'] & tamanho['grande'],
    especie['spinosaurus']
)

# R4: Afiados + Bípede + Chifres + Médio -> Ceratosaurus
# Ceratosaurus: chifre nasal pequeno, porte médio (~6m)
r4 = ctrl.Rule(
    dentes['afiados'] & locomocao['bipede'] & appendice['chifres'] & tamanho['medio'],
    especie['ceratosaurus']
)

# R5: Afiados + Bípede + Crista + Pequeno -> Cryolophosaurus
# Cryolophosaurus: crista transversal única (apelido 'Elvisaurus'), porte médio-pequeno
r5 = ctrl.Rule(
    dentes['afiados'] & locomocao['bipede'] & appendice['crista'] & tamanho['pequeno'],
    especie['cryolophosaurus']
)

# R6: Afiados + Bípede + Dorsal + Médio -> Allosaurus
# Allosaurus: cristas sobre os olhos + porte avantajado mas não gigante (~9m)
r6 = ctrl.Rule(
    dentes['afiados'] & locomocao['bipede'] & appendice['crista'] & tamanho['medio'],
    especie['allosaurus']
)

# R7: Afiados + Bípede + Crista + Médio + Chifres -> Carnotaurus
# Carnotaurus: dois chifres sobre os olhos, porte médio (~7m)
r7 = ctrl.Rule(
    dentes['afiados'] & locomocao['bipede'] & appendice['dorsal'] & tamanho['medio'],
    especie['carnotaurus']
)

# -------------------------------------------------------------------------
# ORNITÓPODES — Herbívoros Bípedes (ou facultativamente bípedes)
# Características de grupo: dentes=achatados, locomocao=bipede/misto
# Diferenciadores: appendice + tamanho
# -------------------------------------------------------------------------

# R8: Achatados + Bípede + Crista + Grande -> Parasaurolophus
# Parasaurolophus: crista craniana tubular longa e curva, porte grande
r8 = ctrl.Rule(
    dentes['achatados'] & locomocao['bipede'] & appendice['crista'] & tamanho['grande'],
    especie['parasaurolophus']
)

# R9: Achatados + Misto + Chifres + Médio -> Iguanodon
# Iguanodon: locomoção facultativa clássica, espinhos nos polegares (não chifres,
# mas mapeamos na mesma região semântica), porte médio-grande
r9 = ctrl.Rule(
    dentes['achatados'] & locomocao['misto'] & appendice['chifres'] & tamanho['medio'],
    especie['iguanodon']
)

# R10: Achatados + Bípede + Dorsal + Médio -> Ouranosaurus
# Ouranosaurus: vela dorsal (como Spinosaurus, mas herbívoro bípede), porte médio
r10 = ctrl.Rule(
    dentes['achatados'] & locomocao['bipede'] & appendice['dorsal'] & tamanho['medio'],
    especie['ouranosaurus']
)

# R11: Achatados + Bípede + Crista + Médio -> Corythosaurus
# Corythosaurus: crista em forma de capacete, porte médio (~9m)
r11 = ctrl.Rule(
    dentes['achatados'] & locomocao['misto'] & appendice['crista'] & tamanho['grande'],
    especie['corythosaurus']
)

# -------------------------------------------------------------------------
# OUTROS HERBÍVOROS — Quadrúpedes
# Características de grupo: dentes=achatados, locomocao=quadrupede
# Diferenciadores: appendice + tamanho
# -------------------------------------------------------------------------

# R12: Achatados + Quadrúpede + Chifres + Grande -> Triceratops
# Triceratops: três chifres icônicos, quadrúpede robusto
r12 = ctrl.Rule(
    dentes['achatados'] & locomocao['quadrupede'] & appendice['chifres'] & tamanho['grande'],
    especie['triceratops']
)

# R13: Achatados + Quadrúpede + Dorsal + Médio -> Stegosaurus
# Stegosaurus: placas dorsais em zigue-zague, porte médio
r13 = ctrl.Rule(
    dentes['achatados'] & locomocao['quadrupede'] & appendice['dorsal'] & tamanho['medio'],
    especie['stegosaurus']
)

# R14: Achatados + Quadrúpede + Chifres + Médio -> Ankylosaurus
# Ankylosaurus: armadura óssea + clava na cauda, sem chifres propriamente,
# mas com projeções ósseas similares; tamanho médio-grande
r14 = ctrl.Rule(
    dentes['achatados'] & locomocao['quadrupede'] & appendice['dorsal'] & tamanho['grande'],
    especie['ankylosaurus']
)

# R15: Achatados + Quadrúpede + Crista + Grande -> Brachiosaurus
# Brachiosaurus: narina elevada (crista de baixo relevo), enorme (22–26m)
r15 = ctrl.Rule(
    dentes['achatados'] & locomocao['quadrupede'] & appendice['crista'] & tamanho['grande'],
    especie['brachiosaurus']
)

print("Base de regras definida: 15 regras, cobrindo todos os dinossauros do roster.")

## Montagem do Sistema de Controle Fuzzy

In [ ]:
# -------------------------------------------------------------------------
# Sistema de controle Mamdani
# O scikit-fuzzy usa 'min' como operação AND e 'max' como OR por padrão,
# o que corresponde ao operador de Mamdani clássico.
# -------------------------------------------------------------------------

fossil_ctrl = ctrl.ControlSystem(rules=[
    r1, r2, r3, r4, r5, r6, r7,    # terópodes
    r8, r9, r10, r11,               # ornitópodes
    r12, r13, r14, r15              # outros herbívoros
])

fossil_sim = ctrl.ControlSystemSimulation(fossil_ctrl)

print("Sistema Mamdani montado. Pronto para inferência.")

# -------------------------------------------------------------------------
# Função auxiliar: traduz o valor defuzzificado para o nome do dinossauro
# -------------------------------------------------------------------------

IDX_TO_DINO = {idx: nome for idx, nome in DINOS}

def identificar(dentes_val, locomocao_val, appendice_val, tamanho_val, verbose=True):
    """
    Roda a inferência fuzzy e retorna o dinossauro mais provável.

    Parâmetros (todos em escala 0–10):
        dentes_val    : 0 = completamente afiados, 10 = completamente achatados
        locomocao_val : 0 = bípede puro, 10 = quadrúpede puro
        appendice_val : 0 = chifres, 5 = crista, 10 = apêndice dorsal
        tamanho_val   : 0 = pequeno (<3m), 10 = grande (>8m)

    Retorna:
        str: nome do dinossauro identificado
    """
    fossil_sim.input['dentes']    = dentes_val
    fossil_sim.input['locomocao'] = locomocao_val
    fossil_sim.input['appendice'] = appendice_val
    fossil_sim.input['tamanho']   = tamanho_val

    fossil_sim.compute()

    resultado_raw = fossil_sim.output['especie']
    idx_mais_proximo = round(resultado_raw)
    idx_mais_proximo = max(0, min(14, idx_mais_proximo))  # clipa no intervalo válido
    dino_identificado = IDX_TO_DINO[idx_mais_proximo]

    if verbose:
        print(f"\n{'='*50}")
        print(f"  FOSSIL LOOKER — RESULTADO DA INFERÊNCIA")
        print(f"{'='*50}")
        print(f"  Entradas:")
        print(f"    Dentes    : {dentes_val:.1f}  ({'afiados' if dentes_val < 3 else 'mistos' if dentes_val < 7 else 'achatados'})")
        print(f"    Locomoção : {locomocao_val:.1f}  ({'bípede' if locomocao_val < 3 else 'misto' if locomocao_val < 7 else 'quadrúpede'})")
        print(f"    Apêndice  : {appendice_val:.1f}  ({'chifres' if appendice_val < 3 else 'crista' if appendice_val < 7 else 'dorsal'})")
        print(f"    Tamanho   : {tamanho_val:.1f}  ({'pequeno' if tamanho_val < 3 else 'médio' if tamanho_val < 7 else 'grande'})")
        print(f"  Saída defuzzificada: {resultado_raw:.3f}")
        print(f"  Índice mais próximo: {idx_mais_proximo}")
        print(f"  ──────────────────────────────────────")
        print(f"  🦕 DINOSSAURO IDENTIFICADO: {dino_identificado.upper()}")
        print(f"{'='*50}\n")

    return dino_identificado

print("Função identificar() pronta.")

## Casos de Teste

Vamos testar 3 casos, com entradas, saída defuzzificada e interpretação comentadas — conforme o requisito da rubrica.

In [ ]:
# =========================================================================
# CASO DE TESTE 1: Spinosaurus
#
# Descrição do espécime:
#   - Dentes cônico-afiados, sem serrilhamento (mais para piscívoro que caçador)
#   - Completamente bípede (postura ereta)
#   - Vela dorsal conspícua
#   - Enorme — maior terópode conhecido
#
# Mapeamento para escalas:
#   dentes    = 1.0  (muito afiados, mas não os mais afiados — dentes cônicos)
#   locomocao = 1.0  (bípede)
#   appendice = 9.0  (apêndice dorsal proeminente)
#   tamanho   = 9.5  (gigantesco)
#
# Resultado esperado: Spinosaurus
# =========================================================================

caso1 = identificar(
    dentes_val=1.0,
    locomocao_val=1.0,
    appendice_val=9.0,
    tamanho_val=9.5
)

In [ ]:
# =========================================================================
# CASO DE TESTE 2: Parasaurolophus
#
# Descrição do espécime:
#   - Dentes extremamente achatados — bateria dental típica de hadrossaurídeo
#   - Locomoção predominantemente bípede
#   - Crista craniana tubular retroprojetada, muito característica
#   - Porte grande (10m)
#
# Mapeamento:
#   dentes    = 9.5  (muito achatados)
#   locomocao = 1.0  (bípede)
#   appendice = 5.0  (crista)
#   tamanho   = 8.5  (grande)
#
# Resultado esperado: Parasaurolophus
# =========================================================================

caso2 = identificar(
    dentes_val=9.5,
    locomocao_val=1.0,
    appendice_val=5.0,
    tamanho_val=8.5
)

In [ ]:
# =========================================================================
# CASO DE TESTE 3: Triceratops
#
# Descrição do espécime:
#   - Dentes achatados — bico de tartaruga + baterias molares
#   - Definitivamente quadrúpede
#   - Três chifres proeminentes + friso craniano (gola)
#   - Grande (~9m, 12 toneladas)
#
# Mapeamento:
#   dentes    = 8.5  (achatados)
#   locomocao = 9.0  (quadrúpede)
#   appendice = 1.0  (chifres)
#   tamanho   = 8.0  (grande)
#
# Resultado esperado: Triceratops
# =========================================================================

caso3 = identificar(
    dentes_val=8.5,
    locomocao_val=9.0,
    appendice_val=1.0,
    tamanho_val=8.0
)

## Exploração Extra: Testando Casos Ambíguos

Uma das vantagens do fuzzy sobre regras crisp é justamente lidar com ambiguidade. Vamos testar um espécime com características "no limite".

In [ ]:
# =========================================================================
# CASO BÔNUS: Iguanodon — o facultativo por excelência
#
# O Iguanodon é o dinossauro que mais quebrou o modelo do MP1:
# ele era facultativamente bípede, o que não se encaixava em nenhuma
# das categorias crisp do sistema anterior.
#
# Aqui, usamos locomocao=5.0 (exatamente no 'misto') para demonstrar
# que o fuzzy captura essa nuance sem exigir uma regra especial.
#
# Mapeamento:
#   dentes    = 8.0  (achatados, herbívoro)
#   locomocao = 5.0  (facultativo — no limite entre bípede e quadrúpede)
#   appendice = 1.5  (espinhos nos polegares, zona de 'chifres')
#   tamanho   = 5.5  (médio)
# =========================================================================

caso_bonus = identificar(
    dentes_val=8.0,
    locomocao_val=5.0,
    appendice_val=1.5,
    tamanho_val=5.5
)

## Comparativo: MP1 (Regras Crisp) vs MP2 (Fuzzy)

### O que mudou e por quê

In [ ]:
comparativo = """
╔══════════════════════════════════════════════════════════════════════╗
║           COMPARATIVO: MP1 (Experta) vs MP2 (scikit-fuzzy)         ║
╠═══════════════════════════╦══════════════════════════════════════════╣
║ Critério                  ║ MP1 (crisp)    │ MP2 (fuzzy Mamdani)   ║
╠═══════════════════════════╬══════════════════════════════════════════╣
║ Entradas                  ║ Booleanas/enum │ Valores contínuos 0–10║
║ Nº de dinossauros         ║ 8              │ 15                     ║
║ Nº de regras              ║ 17 (c/ erros)  │ 15 (sem regras de erro)║
║ Ambiguidade               ║ Sem suporte    │ Graus de pertinência  ║
║ Iguanodon (facultativo)   ║ Não modelável  │ locomocao=5.0         ║
║ Fóssil incompleto         ║ Travado / erro │ Inferência parcial    ║
║ Complexidade de extensão  ║ Alta (+regras) │ Baixa (+mf + 1 regra) ║
║ Expressividade            ║ Binária        │ Gradual / linguística ║
╚═══════════════════════════╩══════════════════════════════════════════╝

  Pontos-chave do comparativo:

  1. COMPLEXIDADE: O MP1 precisava de regras de erro explícitas para
     cobrir combinações inválidas (carnívoro quadrúpede, ornitópode
     com chifre). O MP2 simplesmente não dispara nenhuma regra forte
     para esses casos — o centroide retorna um valor central sem
     classificação confiante. Menos regras, mais gracioso.

  2. EXPRESSIVIDADE: "Dentes levemente afiados" não existia no MP1.
     Aqui, dentes=3.5 faz pertinência simultânea em 'afiados' (0.3)
     e 'mistos' (0.6), refletindo incerteza real de um fóssil.

  3. ESCALABILIDADE: Adicionar um novo dinossauro no MP1 exigia pensar
     em todas as regras de conflito. No MP2, basta adicionar uma MF
     na saída e uma nova regra de antecedente.

  4. TRADE-OFF: O fuzzy perdeu a clareza diagnóstica das regras nítidas.
     No MP1 você sabia exatamente QUAL regra disparou. No MP2,
     múltiplas regras disparam com pesos diferentes — o resultado é
     correto mas menos auditável.
"""

print(comparativo)

## Resumo do Roster

Referência rápida para configurar as entradas do sistema.

In [ ]:
roster = [
    # (nome, dentes, locomocao, appendice, tamanho, grupo)
    ("Velociraptor",    1.0, 1.0, 1.0, 1.0, "Terópode"),
    ("T-Rex",           1.0, 1.0, 1.0, 9.0, "Terópode"),
    ("Spinosaurus",     1.0, 1.0, 9.0, 9.5, "Terópode"),
    ("Ceratosaurus",    1.5, 1.0, 1.5, 5.0, "Terópode"),
    ("Cryolophosaurus", 1.0, 1.0, 5.0, 2.0, "Terópode"),
    ("Allosaurus",      1.0, 1.0, 5.0, 6.0, "Terópode"),
    ("Carnotaurus",     1.0, 1.0, 8.0, 5.0, "Terópode"),
    ("Parasaurolophus", 9.5, 1.0, 5.0, 8.5, "Ornitópode"),
    ("Iguanodon",       8.0, 5.0, 1.5, 5.5, "Ornitópode"),
    ("Ouranosaurus",    9.0, 1.0, 9.0, 5.0, "Ornitópode"),
    ("Corythosaurus",   9.0, 4.0, 5.0, 8.0, "Ornitópode"),
    ("Triceratops",     8.5, 9.0, 1.0, 8.0, "Outro Herbívoro"),
    ("Stegosaurus",     9.0, 9.0, 9.0, 5.0, "Outro Herbívoro"),
    ("Ankylosaurus",    9.0, 9.0, 9.0, 8.0, "Outro Herbívoro"),
    ("Brachiosaurus",   9.0, 9.0, 5.0, 9.5, "Outro Herbívoro"),
]

print(f"{'Dinossauro':<18} {'Dentes':>8} {'Locomoção':>10} {'Apêndice':>10} {'Tamanho':>8}  Grupo")
print("-" * 75)
for nome, d, l, a, t, grupo in roster:
    print(f"{nome:<18} {d:>8.1f} {l:>10.1f} {a:>10.1f} {t:>8.1f}  {grupo}")

---

**Fossil Looker v2.0 — Mini-Projeto 2**  
Controlador Mamdani · 4 entradas · 15 dinossauros · 15 regras · scikit-fuzzy